In [2]:
import os
import warnings
import pandas as pd
from huggingface_hub import hf_hub_download, list_repo_files

# Disable all progress bars and widgets
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'

warnings.filterwarnings('ignore')

# The DRIFT dataset has multiple CSV files with different structures
# Let's explore what's available first
print("Exploring DRIFT dataset structure...")
repo_id = "Hj-Lee/The-DRIFT"

try:
    files = list_repo_files(repo_id, repo_type="dataset")
    csv_files = [f for f in files if f.endswith('.csv')]
    
    print(f"\n✅ Found {len(csv_files)} CSV files:")
    for f in sorted(csv_files)[:20]:  # Show first 20
        print(f"  - {f}")
    
    if len(csv_files) > 20:
        print(f"  ... and {len(csv_files) - 20} more")
        
except Exception as e:
    print(f"Error listing files: {e}")

Exploring DRIFT dataset structure...

✅ Found 160 CSV files:
  - A/drone_1.csv
  - A/drone_10.csv
  - A/drone_11.csv
  - A/drone_12.csv
  - A/drone_13.csv
  - A/drone_14.csv
  - A/drone_15.csv
  - A/drone_16.csv
  - A/drone_17.csv
  - A/drone_18.csv
  - A/drone_2.csv
  - A/drone_3.csv
  - A/drone_4.csv
  - A/drone_5.csv
  - A/drone_6.csv
  - A/drone_7.csv
  - A/drone_8.csv
  - A/drone_9.csv
  - B/drone_1.csv
  - B/drone_10.csv
  ... and 140 more


In [3]:
# Download and explore a specific site's data
# Let's start with site F (mentioned in the error message)

print("Downloading sample file from Site A...")
sample_file = "A/drone_1.csv"

try:
    file_path = hf_hub_download(
        repo_id=repo_id,
        filename=sample_file,
        repo_type="dataset"
    )
    
    # Load and explore the data
    df_sample = pd.read_csv(file_path)
    
    print(f"\n✅ Successfully loaded: {sample_file}")
    print(f"Shape: {df_sample.shape}")
    print(f"\nColumns: {list(df_sample.columns)}")
    print(f"\nFirst few rows:")
    print(df_sample.head())
    
    print(f"\nData types:")
    print(df_sample.dtypes)
    
    print(f"\nBasic statistics:")
    print(df_sample.describe())
    
except Exception as e:
    print(f"Error loading file: {e}")



✅ Successfully loaded: A/drone_1.csv
Shape: (171601, 22)

Columns: ['track_id', 'frame', 'center_x', 'center_y', 'width', 'height', 'angle', 'x1', 'y1', 'x2', 'y2', 'x3', 'y3', 'x4', 'y4', 'confidence', 'class_id', 'site', 'lane', 'preceding_id', 'following_id', 'timestamp']

First few rows:
   track_id  frame     center_x     center_y      width     height     angle  \
0     319.0    297  1788.871338  1163.560913  53.738293  19.964605  3.133297   
1     319.0    298  1788.923096  1163.572876  53.811653  19.981730  3.132453   
2     319.0    299  1788.957031  1163.558960  53.936909  20.020344  3.132043   
3     319.0    300  1789.144775  1163.543335  53.546047  19.868242  3.132135   
4     319.0    301  1789.211670  1163.557129  53.434265  19.823317  3.132755   

            x1           y1           x2  ...           y3           x4  \
0  1761.920410  1153.801880  1762.085938  ...  1173.319946  1815.656738   
1  1761.927124  1153.828369  1762.109741  ...  1173.317383  1815.736450   


In [6]:
# Function to load all data from a specific site
def load_site_data(site_name):
    """Load all CSV files from a specific site (e.g., 'A', 'B', 'F', etc.)"""
    site_files = [f for f in csv_files if f.startswith(f"{site_name}/") and f.endswith('.csv')]
    
    print(f"Loading {len(site_files)} files from Site {site_name}...")
    
    dfs = []
    for file in site_files:
        try:
            file_path = hf_hub_download(
                repo_id=repo_id,
                filename=file,
                repo_type="dataset"
            )
            df = pd.read_csv(file_path)
            df['source_file'] = file  # Track which file the data came from
            dfs.append(df)
            print(f"  ✓ {file}: {df.shape}")
        except Exception as e:
            print(f"  ✗ {file}: {e}")
    
    if dfs:
        combined_df = pd.concat(dfs, ignore_index=True)
        print(f"\n✅ Combined shape: {combined_df.shape}")
        return combined_df
    else:
        print(f"❌ No data loaded from Site {site_name}")
        return None


In [7]:
df = load_site_data('A')

Loading 18 files from Site A...
  ✓ A/drone_1.csv: (171601, 23)
  ✓ A/drone_10.csv: (320113, 23)
  ✓ A/drone_11.csv: (338627, 23)
  ✓ A/drone_12.csv: (40617, 23)
  ✓ A/drone_13.csv: (227173, 23)
  ✓ A/drone_14.csv: (315038, 23)
  ✓ A/drone_15.csv: (264297, 23)
  ✓ A/drone_16.csv: (227770, 23)
  ✓ A/drone_17.csv: (299994, 23)
  ✓ A/drone_18.csv: (81565, 23)
  ✓ A/drone_2.csv: (208280, 23)
  ✓ A/drone_3.csv: (191332, 23)
  ✓ A/drone_4.csv: (279743, 23)
  ✓ A/drone_5.csv: (53264, 23)
  ✓ A/drone_6.csv: (183882, 23)
  ✓ A/drone_7.csv: (342344, 23)
  ✓ A/drone_8.csv: (307870, 23)
  ✓ A/drone_9.csv: (241208, 23)

✅ Combined shape: (4094718, 23)


In [11]:
df_small = df[:1000]

df_small


,track_id,frame,center_x,center_y,width,height,angle,x1,y1,x2,...,x4,y4,confidence,class_id,site,lane,preceding_id,following_id,timestamp,source_file
0,319.0,297,1788.871338,1163.560913,53.738293,19.964605,3.133297,1761.920410,1153.801880,1762.085938,...,1815.656738,1153.356079,0.912315,2.0,Site A,B5,NaN,322.0,07:35:27.012,A/drone_1.csv
1,319.0,298,1788.923096,1163.572876,53.811653,19.981730,3.132453,1761.927124,1153.828369,1762.109741,...,1815.736450,1153.336426,0.912871,2.0,Site A,B5,NaN,322.0,07:35:27.046,A/drone_1.csv
2,319.0,299,1788.957031,1163.558960,53.936909,20.020344,3.132043,1761.894165,1153.806763,1762.085327,...,1815.828735,1153.291626,0.911652,2.0,Site A,B5,NaN,322.0,07:35:27.079,A/drone_1.csv
3,319.0,300,1789.144775,1163.543335,53.546047,19.868242,3.132135,1762.278931,1153.862793,1762.466919,...,1815.822632,1153.356445,0.913222,2.0,Site A,B5,NaN,322.0,07:35:27.112,A/drone_1.csv
4,319.0,301,1789.211670,1163.557129,53.434265,19.823317,3.132755,1762.407959,1153.881958,1762.583252,...,1815.840088,1153.409790,0.914224,2.0,Site A,B5,NaN,322.0,07:35:27.146,A/drone_1.csv
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,322.0,355,1697.579224,1159.893066,53.018009,21.954052,0.008846,1723.990112,1171.104126,1724.184204,...,1670.974243,1170.635132,0.941752,1.0,Site A,B5,319.0,325.0,07:35:28.948,A/drone_1.csv
996,322.0,356,1697.554688,1159.891357,52.953144,21.921225,0.008641,1723.935547,1171.080322,1724.125000,...,1670.984375,1170.622803,0.942948,1.0,Site A,B5,319.0,325.0,07:35:28.981,A/drone_1.csv
997,322.0,357,1697.583740,1159.875000,52.960762,21.919546,0.008533,1723.969604,1171.060303,1724.156616,...,1671.010864,1170.608398,0.942282,1.0,Site A,B5,319.0,325.0,07:35:29.014,A/drone_1.csv
998,322.0,358,1697.594238,1159.891846,52.937408,21.902294,0.008510,1723.968872,1171.067871,1724.155151,...,1671.033325,1170.617432,0.942409,1.0,Site A,B5,319.0,325.0,07:35:29.049,A/drone_1.csv
